# P01 — 1-2 POL(P2) 매터링 펌프 "압력 안 걸림" → 펌프 교체 (2026-10-19)

정리본: [`cases/P01.md`](cases/P01.md). 이 노트북은 그 근거를 다시 만든다 (DB·재생은 캐시가 있으면 다시 안 함).

| 단계 | 내용 |
|---|---|
| 1 | 일지 |
| 2 | 그때 경고 이력 (P2·I2) |
| 3 | 고장 구간 원시 신호 — 가동 지령·트립·흡입 압력 스위치·유량·압력 |
| 4 | 재생 (10-01 ~ 10-24) — 일별 피처 추세 |
| 5 | 고정 조건으로 본 저항 — 순환 25 Hz, 같은 SV 자동 사이클 |
| 6 | 고장 전 전조가 있었나 |
| 7 | 고장 중 모델 화면 상태 |

In [ ]:
import sys; sys.path.insert(0, "..")
import pandas as pd, numpy as np
import matplotlib.pyplot as plt
from review_lib import paths, db, replay, plot
pd.set_option("display.width", 250); pd.set_option("display.max_rows", 200)
FIGS = paths.CHECK_DIRS["pump"] / "figs"; FIGS.mkdir(exist_ok=True)
CACHE = paths.CACHE_DIR / "replay" / "p01_P2I2_1001_1024.pkl"
REPLACE = (pd.Timestamp("2026-10-19 06:12"), pd.Timestamp("2026-10-19 06:34"))   # 흡입 압력 스위치 OFF = 라인 개방
TAGS = sorted({"AL_ETC_Bit", "AL_Line_Bit", "AL_MA_Bit", "AL_Tank_Bit", "Auto_Running", "Auto_Running_PL",
               "Hz_Out_P2", "Pump_BuildUp_P2", "Scale_Out___FT_P2", "Scale_Out___PT_P2", "Scale_Out___TT_P5",
               "Shot_Injection_HD1_P2", "g_s_SV_P2", "Hz_Out_I2", "Pump_BuildUp_I2", "Scale_Out___FT_I2",
               "Scale_Out___PT_I2", "Scale_Out___TT_I3", "g_s_SV_I2"})
AUX = ["P00460", "P00302", "P0004A", "P00440", "P00463", "P00441", "Pump_Run_P2",
       "Scale_Out___FT_P2", "Scale_Out___PT_P2", "Hz_Out_P2", "Scale_Out___FT_I2"]

## 1. 일지

In [ ]:
acts = pd.read_csv(paths.ACTIONS_DIR / "actions_pump.csv")
acts[acts.incident_id == "P01"][["작업ID", "journal", "start_dt", "end_dt", "설비상세", "조치", "확인메모"]]

## 2. 그때 경고 이력 (P2·I2)

10-15 로그 시작부터 P2·I2 묶음으로 "헤드 토출 저항 증가"가 떠 있었다. 교체 뒤(10-19 06:51) I2는 빠지고 P2만 남아 "토출 배관 저항 증가"로 바뀐다.

In [ ]:
ep = pd.read_csv(paths.EPISODES_DIR / "warning_episodes.csv", parse_dates=["onset", "last_detected", "cleared"])
ep[ep.key.isin(["P2", "I2"]) & (ep.onset < "2026-10-26")][["episode_id", "key", "onset", "last_detected", "reasons", "top_features"]]

## 3. 고장 구간 원시 신호

- `P00460` 인버터 FWD(가동 지령) · `Pump_Run_P2` · `P0004A` 트립 · `P00440` 흡입 압력 스위치 (1 = 정상) — **모델 입력 태그가 아니다.** 여기서만 따로 뽑았다.
- P2 압력은 정지 중 65535(0 밑을 부호 없는 16비트로 읽은 값)라 비운다.

In [ ]:
aux = pd.concat([db.fetch_day(d, AUX, group="pump_P2_aux") for d in ["2026-10-18", "2026-10-19"]]).sort_index()
aux["PT_P2"] = aux.Scale_Out___PT_P2.where(aux.Scale_Out___PT_P2 < 60000)
w = aux.loc["2026-10-18 08:20":"2026-10-19 07:00"]
cols = ["P00460", "Pump_Run_P2", "P0004A", "P00440", "Scale_Out___FT_P2", "PT_P2", "Scale_Out___FT_I2"]
w[cols].resample("2min").max().dropna(how="all").query("P00460 > 0 or P00440 < 1")

In [ ]:
fig, axes = plt.subplots(3, 1, figsize=(16, 7), sharex=True)
seg = aux.loc["2026-10-19 05:20":"2026-10-19 07:00"]
axes[0].plot(seg.index, seg.Scale_Out___FT_P2, label="P2 유량"); axes[0].plot(seg.index, seg.Scale_Out___FT_I2, label="I2 유량 (짝)")
axes[0].set_ylabel("유량"); axes[0].legend(loc="upper left")
axes[1].plot(seg.index, seg.PT_P2, color="tab:red"); axes[1].set_ylabel("P2 압력")
axes[2].step(seg.index, seg.P00460, where="post", label="가동 지령 P00460")
axes[2].step(seg.index, seg.P00440 + 0.03, where="post", label="흡입 압력 스위치 P00440")
axes[2].step(seg.index, seg.P0004A - 0.03, where="post", label="트립 P0004A"); axes[2].legend(loc="center right"); axes[2].set_ylim(-0.2, 1.2)
for ax in axes:
    ax.axvspan(pd.Timestamp("2026-10-19 05:28"), pd.Timestamp("2026-10-19 05:52"), color="tab:orange", alpha=0.15)
    ax.axvspan(*REPLACE, color="tab:gray", alpha=0.2)
axes[0].set_title("10-19 새벽 P2 — 주황: 가동 지령 ON·트립 없음·흡입 정상인데 유량 5% (고장) / 회색: 라인 개방(교체)")
fig.tight_layout(); fig.savefig(FIGS / "P01_failure.png", dpi=110, bbox_inches="tight")

## 4. 재생 (10-01 ~ 10-24) — 일별 피처 추세

운영 펌프 모델에 P2·I2 태그를 흘려 사이클마다 판정·표준화 피처(`zn_*`)·원값(`raw_*`)을 얻는다. 처음 한 번은 약 15분(캐시 저장).

In [ ]:
if CACHE.exists():
    C = pd.read_pickle(CACHE)
else:
    parts = []
    for d in pd.date_range("2026-10-01", "2026-10-24"):
        raw = db.fetch_range(d, d + pd.Timedelta(days=1), TAGS, group="pump_HD1_2")
        parts.append(replay.pump_cycles(raw, pumps=["P2", "I2"], progress=False, keep_raw=True))
    C = pd.concat(parts, ignore_index=True)
    C["t"] = pd.to_datetime(C.t_start); C["day"] = (C.t - pd.Timedelta(hours=6)).dt.date
    CACHE.parent.mkdir(parents=True, exist_ok=True); C.to_pickle(CACHE)
A = C[C["mode"] == "자동 운전"]
daily = {}
for p in ["P2", "I2"]:
    daily[p] = A[A.pump == p].groupby("day").agg(
        n=("t", "size"), 알림창비율=("pump_level", lambda s: (s == "알림").mean()),
        PT=("raw_PT", "median"), P0=("raw_P0", "median"), FT_SV=("raw_FT_SV", "median"),
        pump_resid=("raw_pump_resid", "median"), r_phys=("raw_r_phys", "median"),
        z_r=("zn_r_phys", "median"), z_P0=("zn_P0", "median"), z_resid=("zn_pump_resid", "median"))
daily["P2"].round(3)

In [ ]:
d = daily["P2"]; x = pd.to_datetime(pd.Series(d.index))
fig, axes = plt.subplots(3, 1, figsize=(16, 7), sharex=True)
axes[0].plot(x, d.r_phys.values, "o-"); axes[0].axhline(1, color="k", lw=0.5); axes[0].set_ylabel("r_phys\n(실제/기대 압력)")
axes[1].plot(x, d.pump_resid.values, "o-", color="tab:green"); axes[1].axhline(0, color="k", lw=0.5); axes[1].set_ylabel("pump_resid %\n(1 Hz당 유량)")
axes[2].bar(x, d["알림창비율"].values, color="tab:red", alpha=0.6); axes[2].set_ylabel("알림 창 비율")
for ax in axes:
    ax.axvline(pd.Timestamp("2026-10-19"), color="k", ls="--")
axes[0].set_title("P2 일별 (자동 운전 사이클 중앙값) — 점선: 10-19 펌프 교체")
fig.tight_layout(); fig.savefig(FIGS / "P01_trend.png", dpi=110, bbox_inches="tight")

## 5. 고정 조건으로 본 저항 — 교체가 저항을 바꿨나

`r_phys`는 기대 압력을 **회전수 기준 기대 유량(Hz × r)** 으로 계산한다. 그래서 펌프 토출이 줄면 저항이 그대로여도 `r_phys`가 내려간다.
저항만 보려면 실제 유량으로 나눈 K = PT / FT² 를 같은 조건에서 비교한다.

In [ ]:
raw = db.fetch_range("2026-10-01", "2026-10-25", TAGS, group="pump_HD1_2")
r = raw.rename(columns=lambda c: c.replace("Scale_Out___", "").replace("Pump_BuildUp_", "BU_").replace("Hz_Out_", "Hz_"))
hold = r.groupby((r.Hz_P2 != r.Hz_P2.shift()).cumsum()).cumcount()        # 같은 Hz 지령이 이어진 틱 수
circ = r[(r.BU_P2 == 0) & (r.Hz_P2 == 2500) & (r.FT_P2 > 1000) & (r.PT_P2 < 60000) & (hold >= 20)].copy()
circ["day"] = (circ.index - pd.Timedelta(hours=6)).date
circ["K"] = circ.PT_P2 / (circ.FT_P2 / 1000) ** 2
circ.groupby("day").agg(n=("FT_P2", "size"), FT=("FT_P2", "median"), PT=("PT_P2", "median"), K=("K", "median")).round(2)

In [ ]:
# 자동 사이클, 가장 흔한 레시피(SV 3100) 고정
x = A[(A.pump == "P2") & (A.raw_SV_class == 3100)]
x.groupby("day").agg(n=("t", "size"), FT=("raw_FT", "median"), PT=("raw_PT", "median"),
                     K=("raw_K", "median"), FT_Hz=("raw_FT_Hz", "median")).round(3)

## 6. 고장 전 전조가 있었나 — 마지막 자동 운전(10-17)까지 2시간 단위

In [ ]:
f = ["raw_pump_resid", "raw_FT_SV", "zn_t_self", "zn_decel_half_sec", "zn_release_half_sec", "zn_ft_cv", "score"]
y = A[(A.pump == "P2") & A.t.between("2026-10-15 06:00", "2026-10-18")]
y.groupby(y.t.dt.floor("2h"))[f].median().round(3)

## 7. 고장 중 모델 화면 상태

운영 코드(`pump_realtime._tick_state`): 증압 비트가 꺼져 있으면 유량 > `CIRC_MIN_FT`(300)일 때 "대기 순환", 아니면 **"정지"**.
고장 중 유량 58–75 → 화면은 "정지". 자동 사이클이 없으니 판정도 없다. 경고는 10-17 20:15의 "헤드 토출 저항 증가"가 마지막 상태였다.

In [ ]:
seg = aux.loc["2026-10-19 05:28":"2026-10-19 05:52"]
state = np.where(seg.Scale_Out___FT_P2 > 300, "대기 순환", "정지")
pd.Series(state).value_counts()